# Lesson 2 — Count the n-grams

An n-gram model is nothing but counts: for every context, how often did each word come next? Here I build those
counts from a small corpus of everyday sentences.

## The corpus

Sixty short sentences, built in so there's nothing to download. I save the text to `corpus.txt` so the later
lessons can just load it from disk.

In [1]:
CORPUS = '''I drink coffee every morning before work.
The morning train to the city is often late.
She reads a book on the train to work.
We drink tea in the afternoon at the office.
The weather is cold and grey this morning.
He walks to the station in the rain.
On the weekend we walk in the park.
The park is quiet and green in the morning.
I read a good book on the weekend.
Coffee in the morning helps me wake up.
The city is busy and loud during the week.
She takes the late train home from work.
We cook dinner together on the weekend.
The rain falls softly on the quiet street.
He drinks his coffee black and strong.
The office is warm and bright in the winter.
I walk to work when the weather is nice.
The train was late again this morning.
She writes in her journal every evening.
We watch a film together on friday night.
The dog runs happily in the green park.
Morning coffee and a good book make me happy.
The weekend is short but the week is long.
He reads the news on his phone every morning.
The garden is full of flowers in the spring.
I like a quiet morning with hot coffee.
The children play in the park after school.
She cooks a warm dinner on cold evenings.
We travel to the coast in the summer.
The sea is calm and blue in the morning.
Rainy mornings make me want more coffee.
The old cafe on the corner makes good coffee.
He waits for the train on the cold platform.
I finish work and walk home in the evening.
The city lights are bright at night.
She plays the piano softly in the evening.
We read together by the warm fire in winter.
The bakery smells of fresh bread every morning.
I buy fresh bread and coffee on the weekend.
The bus is slower than the morning train.
He enjoys a long walk on sunny weekends.
The library is quiet and full of old books.
She borrows a new book from the library.
Cold weather makes the morning coffee taste better.
We plan a short trip for the long weekend.
The mountains are beautiful in the early morning.
I wake up early and make strong coffee.
The evening is calm after a busy week.
He cooks pasta for dinner on friday.
The rain stopped and the sun came out.
We sit in the park and read our books.
Morning trains are crowded during the week.
She drinks green tea instead of coffee.
The winter mornings are dark and cold.
I love the smell of coffee in the morning.
The weekend market sells fresh fruit and bread.
He reads a chapter of his book before bed.
The quiet street is empty late at night.
We walk the dog in the park every evening.
Good coffee and a good book start my day.'''

In [2]:
with open("corpus.txt", "w") as f:
    f.write(CORPUS)
print("saved corpus.txt —", len(CORPUS), "characters")

saved corpus.txt — 2528 characters


## Splitting text into sentences and words

Sentences end at `.`, `!` or `?`. Words are runs of lower-case letters (apostrophes allowed), so case and
punctuation disappear.

In [3]:
import re, math, random
from collections import Counter, defaultdict

In [4]:
def to_sentences(text):
    return [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]

def to_tokens(sentence):
    return re.findall(r"[a-z']+", sentence.lower())

## Counting what follows what

`build` pads each sentence with `n-1` start markers `<s>` and one end marker `</s>`, so the model also learns
how sentences begin and end. Then it slides a window along the tokens: the first `n-1` tokens are the context and
the last one is the word that followed it.

In [5]:
def build(sents, n):
    """context (n-1 words) -> Counter of the words that follow it."""
    foll = defaultdict(Counter)
    for s in sents:
        toks = ["<s>"] * (n - 1) + to_tokens(s) + ["</s>"]
        for i in range(len(toks) - n + 1):
            foll[tuple(toks[i:i + n - 1])][toks[i + n - 1]] += 1
    return foll

In [6]:
sents = to_sentences(CORPUS)
tokens = [t for s in sents for t in to_tokens(s)]
print("sentences:", len(sents), "| tokens:", len(tokens), "| vocabulary:", len(set(tokens)))

sentences: 60 | tokens: 493 | vocabulary: 169


With `n=2` every context is a single word, written as a 1-tuple like `("the",)`.

In [7]:
foll = build(sents, n=2)                 # context (1 word) -> Counter of next words
print("\nwords that most often follow 'the':", foll[("the",)].most_common(6))
print("words that most often follow 'morning':", foll[("morning",)].most_common(5))
print("words that most often start a sentence:", foll[("<s>",)].most_common(5))


words that most often follow 'the': [('morning', 7), ('weekend', 6), ('park', 5), ('city', 3), ('train', 3), ('rain', 3)]
words that most often follow 'morning': [('</s>', 8), ('train', 2), ('coffee', 2), ('before', 1), ('helps', 1)]
words that most often start a sentence: [('the', 23), ('i', 8), ('we', 8), ('she', 7), ('he', 7)]


These Counters *are* the model — raw evidence of what follows what. "morning" usually ends a sentence
(`</s>` 8 times), and most sentences open with "the". Next lesson turns the counts into probabilities.

## Test yourself

**1. Why add `<s>` and `</s>` to each sentence?**
So the model can learn which words start a sentence (whatever follows `<s>`) and when a sentence should stop
(when `</s>` is predicted). Without them it couldn't begin or end generated text.

**2. For a trigram model, what does `build` use as the context, and how many `<s>` are added?**
The previous two tokens, as a 2-tuple. Two `<s>` are added, so even the first real word has a full
two-token context.

**3. Why does `('the',)` show up as the most common context?**
"the" is by far the most frequent word in English (and in this corpus). Most noun phrases start with it, so lots of
different nouns follow it.

## My notes

The key idea: the whole "model" is a dictionary of Counters — what followed each context. I was surprised that
`</s>` is the most common word after "morning". It shows how much the end marker adds.